# MLOps Capstone – Google Colab Runner
# Unit 8: NYC Green Taxi Tip Prediction
#

# FILE LAYOUT ON COLAB (after setup):
#   /content/project/8/capstone/        → capstone_flow.py, capstone_lib.py, inference.py
#   /content/project/6/                 → green_taxi_drift_lib.py
#   /content/project/8/capstone/data/   → TLC parquet files (downloaded)
#   /content/mlruns/                    → MLflow tracking DB + artifacts

In [1]:
# Installs all libraries required by capstone_flow.py, capstone_lib.py, and inference.py

import subprocess
subprocess.run(['pip', 'install', '-q', 'mlflow==3.8', 'metaflow', 'nannyml', 'optuna',
                'scikit-learn', 'pyarrow', 'xgboost', 'pandas', 'numpy'], check=True)


CompletedProcess(args=['pip', 'install', '-q', 'mlflow==3.8', 'metaflow', 'nannyml', 'optuna', 'scikit-learn', 'pyarrow', 'xgboost', 'pandas', 'numpy'], returncode=0)

In [2]:
import os
import sys
import shutil
import time
import subprocess
import urllib.request

import mlflow
import metaflow
import nannyml
import optuna
import pandas as pd

from mlflow import MlflowClient
from google.colab import files

print('mlflow  :', mlflow.__version__)
print('metaflow:', metaflow.__version__)
print('nannyml :', nannyml.__version__)
print('optuna  :', optuna.__version__)
print('ALL OK')


mlflow  : 3.8.0
metaflow: 2.19.28
nannyml : 0.13.1
optuna  : 4.8.0
ALL OK


In [3]:
# Clone Repo + Setup Paths

# After this cell:
#   - capstone_flow.py, capstone_lib.py, inference.py are at /content/project/8/capstone/
#   - green_taxi_drift_lib.py (Unit 6 shared lib) is at /content/project/6/ AND
#     copied into /content/project/8/capstone/ so Metaflow subprocesses find it.
#   - cwd is set to CAPSTONE so relative --batch-path / --reference-path paths work.

REPO_URL  = 'https://github.com/arielshtangel/OU-22971-MLOps.git'
CAPSTONE  = '/content/project/8/capstone'
DRIFT_LIB = '/content/project/6'

os.system('rm -rf /content/project')
os.system(f'git clone {REPO_URL} /content/project')

os.makedirs(f'{CAPSTONE}/data', exist_ok=True)

# Copy green_taxi_drift_lib.py into capstone/ so Metaflow subprocesses can find it
shutil.copy(f'{DRIFT_LIB}/green_taxi_drift_lib.py', f'{CAPSTONE}/green_taxi_drift_lib.py')

# Environment for all subprocess calls
os.environ['PYTHONPATH'] = f'{CAPSTONE}:{DRIFT_LIB}'
os.environ['USER'] = 'student'
sys.path.insert(0, CAPSTONE)
sys.path.insert(0, DRIFT_LIB)
os.chdir(CAPSTONE)

print('Capstone files:', os.listdir(CAPSTONE))
print('Working dir   :', os.getcwd())
print('Setup complete — ready to run.')


Capstone files: ['capstone_lib.py', 'inference.py', 'green_taxi_drift_lib.py', 'data', 'capstone_flow.py']
Working dir   : /content/project/8/capstone
Setup complete — ready to run.


In [4]:
# Download TLC Green Taxi Data

# Files downloaded and their roles in the demo runs:
#   green_tripdata_2023-01.parquet → REFERENCE dataset (January, stable baseline)
#   green_tripdata_2023-04.parquet → BATCH for Run 1 (small drift; no retrain expected)
#   green_tripdata_2023-10.parquet → BATCH for Run 2 & 3 (seasonal drift; retrain expected)

DATA = f'{CAPSTONE}/data'
BASE = 'https://d37ci6vzurychx.cloudfront.net/trip-data'

os.system(f'wget -q --show-progress -O {DATA}/green_tripdata_2023-01.parquet {BASE}/green_tripdata_2023-01.parquet')
os.system(f'wget -q --show-progress -O {DATA}/green_tripdata_2023-04.parquet {BASE}/green_tripdata_2023-04.parquet')
os.system(f'wget -q --show-progress -O {DATA}/green_tripdata_2023-10.parquet {BASE}/green_tripdata_2023-10.parquet')

print('\nDone. Files:')
os.system(f'ls -lh {DATA}/')



Done. Files:


0

In [5]:
# Start MLflow Tracking Server (background)

os.makedirs('/content/mlruns', exist_ok=True)

subprocess.run(['pkill', '-f', 'mlflow'], capture_output=True)
time.sleep(2)

mlflow_proc = subprocess.Popen(
    ['mlflow', 'server',
     '--host', '127.0.0.1',
     '--port', '5000',
     '--backend-store-uri', 'sqlite:////content/mlruns/mlflow.db',
     '--default-artifact-root', '/content/mlruns/artifacts'],
    stdout=open('/content/mlruns/mlflow.log', 'w'),
    stderr=subprocess.STDOUT
)

for i in range(30):
    time.sleep(1)
    try:
        urllib.request.urlopen('http://127.0.0.1:5000')
        print(f'MLflow server running (took {i+1}s)')
        break
    except:
        print(f'  waiting... {i+1}s', end='\r')
else:
    print('Still not responding. Logs:')
    with open('/content/mlruns/mlflow.log') as f:
        print(f.read())


MLflow server running (took 30s)


In [6]:
# Demo Run 1: Baseline (No Retrain, No Promotion)

# Flow steps executed in this run:
#   Step A – start        : create MLflow run, log flow parameters
#   Step A – load_data    : load Jan-2023 (reference) + Apr-2023 (batch)
#   Step B – integrity_gate : Layer 1 hard rules + Layer 2 NannyML soft checks
#   Step C – feature_engineering : time/location/numeric features; feature_spec.json logged
#   Step D – load_champion : bootstrap initial model OR load existing @champion
#   Step E – model_gate   : evaluate champion on batch;
#                           --retrain-rmse-threshold 0.99 → RMSE increase < 99%
#                           → retrain_recommended = false
#   Step G – candidate_acceptance : no candidate → action = "no_action"
#   end   – finalise MLflow run status = FINISHED
#
# Note: --retrain-rmse-threshold 0.99 sets an intentionally high threshold
# so that even seasonal drift does NOT trigger retraining in this run.

os.chdir(CAPSTONE)
os.system(
    'python capstone_flow.py run'
    ' --reference-path data/green_tripdata_2023-01.parquet'
    ' --batch-path     data/green_tripdata_2023-04.parquet'
    ' --retrain-rmse-threshold 0.99'
    ' --tracking-uri http://127.0.0.1:5000'
)


0

In [7]:
# Demo Run 2: Retrain + Promotion (Automatic)

# Flow steps executed in this run:
#   Step A – start + load_data : load Jan-2023 (reference) + Oct-2023 (batch)
#   Step B – integrity_gate    : hard rules + NannyML soft checks on Oct batch
#   Step C – feature_engineering : apply same feature pipeline to both splits
#   Step D – load_champion     : load existing @champion from registry
#   Step E – model_gate        : --retrain-rmse-threshold 0.001
#                               (very low threshold → any RMSE degradation
#                               triggers retrain) → retrain_recommended = true
#   Step F – retrain           : rolling window (reference + Oct batch);
#                               Optuna 15-trial hyperparameter search;
#                               candidate registered (role=candidate, validation_status=pending)
#   Step G – candidate_acceptance : P1–P4 gate evaluation:
#     P1 – evaluation valid (rmse_candidate not NaN)
#     P2 – candidate beats champion by ≥ min_improvement (default 1%)
#     P3 – candidate does not regress reference slice by > 10%
#     P4 – no hard integrity failure
#     → promote = true → @champion alias flipped to new version
#       old champion tagged role=previous_champion, demoted_at=<utc>
#       new champion tagged role=champion, promoted_at=<utc>,
#                             promotion_reason=candidate_beat_champion,
#                             validation_status=approved
#   decision.json logged with action = "promote"

os.chdir(CAPSTONE)
os.system(
    'python capstone_flow.py run'
    ' --reference-path data/green_tripdata_2023-01.parquet'
    ' --batch-path     data/green_tripdata_2023-10.parquet'
    ' --retrain-rmse-threshold 0.001'
    ' --tracking-uri http://127.0.0.1:5000'
)


0

In [8]:
# Demo Run 3 Step A: Inject Failure into retrain step

# The injection targets the first line inside the retrain step body so that
# the failure is reproducible and clearly visible in the Metaflow output.

flow_path = f'{CAPSTONE}/capstone_flow.py'

with open(flow_path, 'r') as f:
    content = f.read()

INJECTION_LINE = '        raise RuntimeError("injected failure for demo run 3")  # [DEMO3]\n'
TARGET = '        import optuna\n        from capstone_lib import rmse'

if '[DEMO3]' not in content:
    if TARGET in content:
        patched = content.replace(TARGET, INJECTION_LINE + TARGET, 1)
        with open(flow_path, 'w') as f:
            f.write(patched)
        print('Failure injected into retrain step.')
    else:
        print('TARGET string not found.')
else:
    print('Failure already injected.')


Failure injected into retrain step.


In [9]:
# Demo Run 3 Step B: Run Flow (Expect Failure at retrain)

# The '|| true' ensures the notebook cell itself does not fail so we can continue

os.chdir(CAPSTONE)
os.system(
    'python capstone_flow.py run'
    ' --reference-path data/green_tripdata_2023-01.parquet'
    ' --batch-path     data/green_tripdata_2023-10.parquet'
    ' --retrain-rmse-threshold 0.001'
    ' --tracking-uri http://127.0.0.1:5000 || true'
)
print('\n--- Flow failed as expected ---')



--- Flow failed as expected ---


In [10]:
# Demo Run 3 Step C: Remove Injected Failure

with open(flow_path, 'r') as f:
    content = f.read()

fixed = '\n'.join(line for line in content.splitlines() if '[DEMO3]' not in line)

with open(flow_path, 'w') as f:
    f.write(fixed)

print('Injected failure removed. Ready to resume.')


Injected failure removed. Ready to resume.


In [11]:
# Demo Run 3 Step D: Resume from Failed Step

os.chdir(CAPSTONE)
os.system('python capstone_flow.py resume retrain')


0

In [12]:
# Batch Inference

# inference.py:
#   1. Loads @champion model from the registry
#      (Design Doc §Step D – "models:/<model_name>@champion")
#   2. Applies the SAME feature engineering pipeline used at training time
#      (Design Doc §Feature engineering – "reused consistently for training,
#       evaluation, and inference")
#   3. Runs predictions on the October batch
#   4. Saves predictions.parquet locally
#   5. Logs predictions.parquet as an MLflow artifact for full lineage
#      (Design Doc §P1 – "evaluation dataset is logged (lineage)")
#   6. Logs rmse_inference if labels are present

os.chdir(CAPSTONE)
os.system(
    'python inference.py'
    ' --batch-path   data/green_tripdata_2023-10.parquet'
    ' --tracking-uri http://127.0.0.1:5000'
)


0

In [13]:
# Verify Predictions File

pred_files = [f for f in os.listdir(CAPSTONE) if 'pred' in f and f.endswith('.parquet')]

if pred_files:
    df = pd.read_parquet(os.path.join(CAPSTONE, pred_files[0]))
    print(f'{pred_files[0]}: {len(df):,} rows')
    print(df.head())
else:
    print('No predictions file found — check Cell 12 output.')


predictions.parquet: 41,545 rows
   VendorID  RatecodeID  PULocationID  DOLocationID  passenger_count  \
0       2.0         1.0         166.0          74.0              1.0   
1       2.0         1.0          74.0         263.0              1.0   
2       2.0         1.0          74.0         236.0              1.0   
3       2.0         1.0         129.0         157.0              1.0   
4       2.0         1.0         152.0         244.0              2.0   

   trip_distance  fare_amount  extra  mta_tax  tolls_amount  ...  \
0           1.45         12.1    1.0      0.5           0.0  ...   
1           2.26         11.4    1.0      0.5           0.0  ...   
2           2.14         13.5    1.0      0.5           0.0  ...   
3           2.18         13.5    1.0      0.5           0.0  ...   
4           1.22          9.3    1.0      0.5           0.0  ...   

   lpep_pickup_month  lpep_pickup_weekday  lpep_pickup_hour  \
0               10.0                  6.0               0.0   

In [14]:
# Inspect MLflow Runs (Programmatic Summary)

# This cell provides a programmatic summary of the last 6 runs so reviewers
# can quickly verify that all three required demo runs are present and have
# the correct outcome tags:
#   Run 1 → retrain_recommended=false, promotion_recommended=false
#   Run 2 → retrain_recommended=true,  promotion_recommended=true
#   Run 3 → retrain_recommended=true,  promotion_recommended=true (after resume)

mlflow.set_tracking_uri('http://127.0.0.1:5000')
client = MlflowClient()

exp = client.get_experiment_by_name('8_capstone')
if exp:
    runs = client.search_runs(exp.experiment_id, order_by=['start_time DESC'], max_results=6)
    print(f'{"Run Name":<32} {"Status":<10} {"retrain_rec":<14} {"promotion_rec"}')
    print('-' * 75)
    for r in runs:
        name    = r.data.tags.get('mlflow.runName', r.info.run_id[:8])
        retrain = r.data.tags.get('retrain_recommended', '-')
        promote = r.data.tags.get('promotion_recommended', '-')
        print(f'{name:<32} {r.info.status:<10} {retrain:<14} {promote}')
else:
    print('No 8_capstone experiment found — run Demo Run 1: Baseline cell first.')


Run Name                         Status     retrain_rec    promotion_rec
---------------------------------------------------------------------------
capstone_green_tripdata_2023-10  FINISHED   false          false
retrain_candidate                FINISHED   -              -
capstone_green_tripdata_2023-10  FINISHED   true           true
bootstrap_train                  FINISHED   -              -
capstone_green_tripdata_2023-04  FINISHED   false          false


In [15]:
# Inspect Model Registry

try:
    versions = client.search_model_versions("name='green_taxi_tip_model'")
    print(f'{"Version":<10} {"Role":<22} {"Status":<12} {"Aliases"}')
    print('-' * 60)
    for v in sorted(versions, key=lambda x: int(x.version)):
        role    = v.tags.get('role', '-')
        aliases = getattr(v, 'aliases', [])
        print(f'{v.version:<10} {role:<22} {v.current_stage:<12} {aliases}')
except Exception as e:
    print(f'Registry check failed: {e}')


Version    Role                   Status       Aliases
------------------------------------------------------------
1          previous_champion      None         []
2          champion               None         []


In [16]:
# Download MLflow Database (for submission)

# The SQLite file mlflow.db contains the complete record of:
#   • All experiment runs (metrics, params, tags, status)
#   • All model versions and aliases in the Model Registry
#   • All logged artifact paths (decision.json, feature_spec.json,
#     nannyml_soft.json, predictions.parquet, etc.)

files.download('/content/mlruns/mlflow.db')
print('mlflow.db downloaded.')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

mlflow.db downloaded.
